<a href="https://colab.research.google.com/github/eternitvy/ML_26/blob/main/JS09/JS09_TUGAS1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
# ==========================================
# 0. INSTALL LIBRARY (Jalankan jika belum ada)
# ==========================================
import subprocess
import sys

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

try:
    import pandas as pd
    import annoy
    import faiss
    import hnswlib
except ImportError:
    print("Menginstall library yang kurang...")
    install("pandas")
    install("annoy")
    install("faiss-cpu")
    install("hnswlib")
    import pandas as pd
    import annoy
    import faiss
    import hnswlib

import numpy as np
import time
from annoy import AnnoyIndex

# ==========================================
# 1. DEFINISI SKENARIO TUGAS LAB 1
# ==========================================
scenarios = [
    (1_000, 2, 'euclidean'),
    (1_000, 2, 'cosine'),
    (1_000_000, 5, 'euclidean'),
    (1_000_000, 5, 'cosine')
]

all_results = []
k = 10 # Jumlah tetangga terdekat (k-NN)

print("=== MEMULAI PROSES BENCHMARK ANN ===")

for n_data, dim, metric in scenarios:
    print(f"Sedang memproses skenario: {n_data} data / {dim}D / {metric}...")

    # Generate data random
    np.random.seed(42)
    X = np.random.random((n_data, dim)).astype(np.float32)
    query = np.random.random((1, dim)).astype(np.float32)

    # --------------------------------------
    # 1. ANNOY
    # --------------------------------------
    ann_metric = 'euclidean' if metric == 'euclidean' else 'angular'
    ann_index = AnnoyIndex(dim, ann_metric)
    for i in range(n_data):
        ann_index.add_item(i, X[i])

    start_b = time.time()
    ann_index.build(10) # 10 trees
    ann_build = time.time() - start_b

    start_q = time.time()
    _ = ann_index.get_nns_by_vector(query[0], k)
    ann_query = time.time() - start_q

    # --------------------------------------
    # 2. FAISS
    # --------------------------------------
    if metric == 'euclidean':
        faiss_index = faiss.IndexFlatL2(dim)
    else:
        faiss_index = faiss.IndexFlatIP(dim) # Inner product untuk cosine

    start_b = time.time()
    faiss_index.add(X)
    faiss_build = time.time() - start_b

    start_q = time.time()
    _, _ = faiss_index.search(query, k)
    faiss_query = time.time() - start_q

    # --------------------------------------
    # 3. HNSW (hnswlib)
    # --------------------------------------
    hnsw_space = 'l2' if metric == 'euclidean' else 'cosine'
    hnsw_index = hnswlib.Index(space=hnsw_space, dim=dim)
    hnsw_index.init_index(max_elements=n_data, ef_construction=200, M=16)

    start_b = time.time()
    hnsw_index.add_items(X)
    hnsw_build = time.time() - start_b

    hnsw_index.set_ef(50)
    start_q = time.time()
    _, _ = hnsw_index.knn_query(query, k=k)
    hnsw_query = time.time() - start_q

    # Simpan hasil perbandingan ke format string ringkas
    row_data = {
        "Jumlah Data/Dimensi": f"{n_data}/{dim}D ({metric.capitalize()})",
        "ANNOY": f"Build: {ann_build:.4f}s\nQuery: {ann_query*1000:.3f}ms",
        "FAISS": f"Build: {faiss_build:.4f}s\nQuery: {faiss_query*1000:.3f}ms",
        "HNSW": f"Build: {hnsw_build:.4f}s\nQuery: {hnsw_query*1000:.3f}ms"
    }
    all_results.append(row_data)

# ==========================================
# 2. CETAK HASIL DALAM BENTUK TABEL
# ==========================================
df_tugas = pd.DataFrame(all_results)

print("\n" + "="*40)
print("HASIL AKHIR TABEL TUGAS LAB 1")
print("="*40)
print(df_tugas.to_markdown(index=False))

=== MEMULAI PROSES BENCHMARK ANN ===
Sedang memproses skenario: 1000 data / 2D / euclidean...
Sedang memproses skenario: 1000 data / 2D / cosine...
Sedang memproses skenario: 1000000 data / 5D / euclidean...
Sedang memproses skenario: 1000000 data / 5D / cosine...

HASIL AKHIR TABEL TUGAS LAB 1
| Jumlah Data/Dimensi    | ANNOY           | FAISS          | HNSW             |
|:-----------------------|:----------------|:---------------|:-----------------|
| 1000/2D (Euclidean)    | Build: 0.1285s  | Build: 0.0001s | Build: 0.3634s   |
|                        | Query: 0.084ms  | Query: 0.070ms | Query: 0.090ms   |
| 1000/2D (Cosine)       | Build: 0.1186s  | Build: 0.0000s | Build: 0.1826s   |
|                        | Query: 0.072ms  | Query: 0.059ms | Query: 0.075ms   |
| 1000000/5D (Euclidean) | Build: 18.5246s | Build: 0.0146s | Build: 160.4109s |
|                        | Query: 0.101ms  | Query: 8.071ms | Query: 0.124ms   |
| 1000000/5D (Cosine)    | Build: 25.8045s | Build: 0.00